# Del interferograma a la fase continua: phase stepping y desenrollado

**Objetivo.** Recorrer, de la teoría a la ejecución, la cadena de análisis de franjas que
convierte un conjunto de interferogramas en un mapa de **fase continua** y, con él, en la
**fracción de franja** que alimenta la determinación dimensional (L1). Son dos etapas que
ocurren **una tras otra**:

1. **Desplazamiento de fase** (*phase stepping* / PSI): de un *stack* de $N$ interferogramas
   con fase de referencia desplazada, a la **fase envuelta** $\varphi(x,y)\in(-\pi,\pi]$.
2. **Desenrollado** (*unwrapping*): de la fase envuelta a la **fase continua** $\Phi(x,y)$,
   sumando los múltiplos de $2\pi$ que faltan.

Cada bloque de código **importa y ejecuta las funciones reales de `interflib`**; no reimplementa
nada. El banco de pruebas es el simulador por capas `interflib.simulation`, que además entrega la
**verdad de terreno** con la que se contrasta cada resultado.

**Alcance.** Datos simulados (aún no hay medición real). El método de Fourier de una sola toma
(portadora espacial), alternativa a PSI cuando solo se dispone de una imagen, queda fuera de este
cuaderno.

## 0. Preparación

Se importan las funciones públicas de las dos etapas y del simulador. La convención de signos y
unidades es la de la librería: longitudes de onda y OPD en nanómetros; una altura de superficie
$h$ produce $\mathrm{OPD}=2h$ (doble paso) y una fase $\varphi=(2\pi/\lambda)\,\mathrm{OPD}$. Los
símbolos siguen la nomenclatura del proyecto: $\varphi$ fase envuelta, $\Phi$ fase continua,
$I_0$ fondo, $V$ visibilidad, $F$ fracción de franja, $\delta_k$ paso de fase.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from interflib.simulation import (
    SceneGeometry, PhaseModel, Illumination, SensorModel,
    synthesize_interferogram, fringe_fraction_of_length,
)
from interflib.fringe.phase_shifting import (
    least_squares_psa, three_step, four_step, hariharan_five_step,
    fraction_from_phase,
)
from interflib.fringe.unwrap import (
    wrap_phase, find_residues, unwrap_itoh,
    unwrap_quality_guided, unwrap_goldstein,
)

plt.rcParams.update({"figure.dpi": 110, "image.cmap": "gray", "font.size": 10})
np.set_printoptions(precision=4, suppress=True)

# --- Constantes de la escena ---------------------------------------------------------
WAVELENGTH_NM = 632.8       # He-Ne rojo estabilizado (Renishaw XL-80)
CENTRAL_LENGTH_NM = 1.0e7   # bloque patron de 10 mm de longitud nominal
STEPS = np.arange(5) * np.pi / 2.0   # pasos nominales: 0, pi/2, pi, 3pi/2, 2pi

# --- Geometrias ----------------------------------------------------------------------
#  con bloque (cara ISO 3650 ~ 9 x 30 mm para longitudes <= 10 mm)
geometry = SceneGeometry(resolution=(240, 320), fov_mm=25.0,
                         block_width_mm=9.0, block_height_mm=30.0)
#  sin bloque (campo continuo, para aislar el comportamiento de los algoritmos)
geom_plain = SceneGeometry(resolution=(240, 320), fov_mm=25.0,
                           block_width_mm=0.0, block_height_mm=0.0)

# --- Iluminacion y sensores ----------------------------------------------------------
illumination = Illumination(background=0.5, vignette=0.12,
                            visibility=0.85, visibility_falloff=0.15, psf_sigma_px=0.0)
#  poca luz y fuerte caida del contraste hacia los bordes: alli la modulacion se acerca
#  al ruido (independiente por cuadro) y la fase deja de ser fiable -> aparecen residuos
low_light = Illumination(background=0.02, beam_waist_frac=0.55, vignette=0.5,
                         visibility=0.85, visibility_falloff=0.95, psf_sigma_px=0.5)

clean_sensor = SensorModel(shot_noise=False, read_noise_e=0.0, bit_depth=16, saturate=False)
camera = SensorModel.cs505mu()   # camara real: 12 bits, ruido de disparo + lectura


def make_stack(applied_steps, *, geometry, illumination, sensor, use_irradiance,
               length_nm=CENTRAL_LENGTH_NM, fringes=(1.0, 6.0),
               tilt_mismatch=(0.0, 0.0), flatness_pv=0.0, seed0=0):
    # Synthesise a phase-shifted stack; returns (frames, ground-truth of last frame).
    frames, truth = [], None
    for k, delta in enumerate(applied_steps):
        model = PhaseModel(
            wavelength_nm=WAVELENGTH_NM, central_length_nm=length_nm,
            fringes_x=fringes[0], fringes_y=fringes[1], phase_shift_rad=float(delta),
            block_tilt_mismatch_x=tilt_mismatch[0], block_tilt_mismatch_y=tilt_mismatch[1],
            block_flatness_pv_nm=flatness_pv,
        )
        s = synthesize_interferogram(geometry, model, illumination, sensor, seed=seed0 + k)
        frames.append(s.irradiance if use_irradiance else s.image.astype(float))
        truth = s
    return np.array(frames), truth


def show(img, title="", cmap="gray", ax=None, colorbar=False, **kw):
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 3))
    im = ax.imshow(img, cmap=cmap, **kw)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    if colorbar:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax


def demeaned_rms_error(recovered, truth_phase, mask=None):
    # RMS of the wrapped phase error after removing the constant piston offset.
    err = wrap_phase(recovered - truth_phase)
    if mask is not None:
        err = err[mask]
    err = wrap_phase(err - np.angle(np.mean(np.exp(1j * err))))
    return float(np.sqrt(np.mean(err ** 2)))

print("interflib cargado. lambda =", WAVELENGTH_NM, "nm ; L =", CENTRAL_LENGTH_NM / 1e6, "mm")

## 1. El problema: de la intensidad a la fase

Un interferograma de dos haces obedece, píxel a píxel,

$$I(x,y) = a(x,y) + b(x,y)\,\cos\varphi(x,y),$$

donde $a$ es el **fondo** (nuestro $I_0$) y $b$ la **amplitud de modulación** (nuestro
$I_0\,V$); la visibilidad es $V=b/a$. Toda la información métrica está en $\varphi$, pero
recuperarla de una sola toma es un **problema inverso mal planteado**: en cada píxel hay tres
incógnitas ($a$, $b$, $\varphi$) y una sola ecuación, y además el coseno es par
($\cos\varphi=\cos(-\varphi)$, ambigüedad de signo) y periódico (ambigüedad de $2\pi$)
[Servín, Quiroga & Padilla, *Fringe Pattern Analysis for Optical Metrology* (2014), §1.1].

La salida es introducir una **portadora** que separe espectralmente la señal del fondo. El
desplazamiento de fase la introduce en el **tiempo**: se toman varias imágenes desplazando la
fase de referencia en pasos conocidos $\delta_k$, y ese barrido convierte el problema en uno
**lineal y sobredeterminado**.

## 2. Desplazamiento de fase: teoría

Con la fase de referencia desplazada en $\delta_k$, el cuadro $k$ registra

$$I_k = a + b\,\cos(\varphi + \delta_k), \qquad k = 0,\dots,N-1.$$

Desarrollando $\cos(\varphi+\delta_k)=\cos\varphi\cos\delta_k-\sin\varphi\sin\delta_k$, cada
cuadro es una **ecuación lineal** en las tres incógnitas $a$, $b\cos\varphi$ y $b\sin\varphi$:

$$I_k = \underbrace{a}_{x_0} + \underbrace{(b\cos\varphi)}_{x_1}\cos\delta_k
        + \underbrace{(b\sin\varphi)}_{x_2}(-\sin\delta_k).$$

Con $N\ge 3$ pasos que separen fondo, seno y coseno, el sistema se resuelve por **mínimos
cuadrados** con una matriz de diseño común a todos los píxeles (una sola pseudoinversa demodula
el campo entero) [Bruning *et al.*, *Appl. Opt.* **13**, 2693 (1974); Greivenkamp,
*Opt. Eng.* **23**, 350 (1984)]:

$$\varphi = \operatorname{atan2}(b\sin\varphi,\; b\cos\varphi),\qquad
  b = \sqrt{(b\cos\varphi)^2+(b\sin\varphi)^2},\qquad V=\frac{b}{a}.$$

**Algoritmos de pasos fijos.** Para pasos de $\pi/2$ el sistema de 3 o 4 cuadros queda
exactamente determinado y coincide con las formas cerradas clásicas. El de **cinco pasos de
Schwider–Hariharan** ($\delta_k=0,\pi/2,\pi,3\pi/2,2\pi$),

$$\varphi=\operatorname{atan2}\!\big(2(I_4-I_2),\; I_1-2I_3+I_5\big),$$

promedia el quinto cuadro redundante contra el primero; su función de transferencia tiene un
**cero doble** en la portadora, de modo que un error **lineal** en el tamaño del paso (p. ej. un
PZT mal calibrado) se cancela a primer orden [Hariharan, Oreb & Eiju, *Appl. Opt.* **26**, 2504
(1987)]. Es el estimador de cinco cuadros de referencia.

La fase recuperada sale **envuelta** en $(-\pi,\pi]$; reconstruir la continua es la segunda
etapa (§6).

## 3. Generación del *stack* con el simulador

Se sintetiza un *stack* de **cinco** interferogramas de un bloque de 10 mm sobre su platina,
desplazando la fase global con el gancho `PhaseModel.phase_shift_rad` $=\delta_k$. El simulador
devuelve, además de la imagen, la **verdad de terreno**: el mapa de OPD, la fase envuelta, la
máscara del bloque y la fracción `fraction_truth` $=\operatorname{frac}(2L/\lambda)$.

In [ ]:
frames_clean, truth = make_stack(STEPS, geometry=geometry, illumination=illumination,
                                 sensor=clean_sensor, use_irradiance=True, fringes=(0.0, 6.0))

print("stack:", frames_clean.shape, "(N, alto, ancho)")
print("fraccion verdadera F = frac(2L/lambda) =", round(truth.fraction_truth, 6))

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for k in range(5):
    show(frames_clean[k], title=fr"$\delta_{{{k}}} = {k}\pi/2$", ax=axes[k])
show(truth.opd_nm, title="OPD verdad (nm)", cmap="viridis", ax=axes[5], colorbar=True)
plt.tight_layout(); plt.show()

**Lectura de la figura.** Los cinco cuadros son la misma escena con la fase de referencia
corrida en $\pi/2$: las franjas se **desplazan** de un cuadro al siguiente (un periodo completo
entre el primero y el quinto). La banda vertical central es el **bloque**, algo más brillante o
más oscuro que la platina según su fracción de franja. El mapa de OPD (derecha) muestra la rampa
de *tilt* que hace de portadora y el **escalón** del bloque respecto a la platina; es lo que la
demodulación debe recuperar.

## 4. Demodulación: recuperar $\varphi$, $I_0$, $b$ y $V$

Se aplica el algoritmo de cinco pasos a las funciones reales de la librería. El resultado trae
la fase envuelta y, gratis, el **mapa de modulación** $b$: la métrica de confianza por píxel que
guiará el desenrollado.

In [ ]:
psi = hariharan_five_step(frames_clean)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
show(psi.phase, title=r"fase envuelta $\varphi$", cmap="twilight", ax=axes[0], colorbar=True)
show(psi.background, title=r"fondo $a = I_0$", cmap="viridis", ax=axes[1], colorbar=True)
show(psi.modulation, title=r"modulacion $b$ (calidad)", cmap="viridis", ax=axes[2], colorbar=True)
show(psi.visibility, title=r"visibilidad $V=b/a$", cmap="viridis", ax=axes[3], colorbar=True)
plt.tight_layout(); plt.show()

err = demeaned_rms_error(psi.phase, truth.phase)
print("RMS |phi_recuperada - phi_verdad| (sin pluma) =", f"{err:.2e}", "rad")

**Lectura de la figura.** La **fase envuelta** (izquierda, mapa cíclico) muestra la rampa de
*tilt* saltando de $+\pi$ a $-\pi$ en cada franja, con el escalón del bloque visible como un
corrimiento del patrón en la banda central. El **fondo** $I_0$ reproduce el perfil gaussiano de
iluminación con su viñeteo; la **modulación** $b$ decae hacia los bordes (menos contraste), y la
**visibilidad** $V=b/a$ resume ese contraste normalizado. La demodulación recupera la fase con un
error a nivel de precisión de máquina (campo limpio, sin ruido): la etapa PSI, sobre datos
ideales, es exacta.

## 5. Los algoritmos y la robustez a la desintonía

Sobre datos limpios los cuatro estimadores (3 pasos, 4 pasos, mínimos cuadrados general y
Hariharan-5) coinciden. La diferencia aparece cuando el paso está **mal calibrado**: los pasos
reales son $\delta_k(1+\varepsilon)$ pero el algoritmo asume los nominales. Se compara el error
del estimador de cuatro pasos con el de Hariharan-5 barriendo $\varepsilon$, sobre un campo
**sin bloque** para aislar la rampa.

In [ ]:
# En datos limpios los cuatro estimadores dan la misma fase:
r_ls = least_squares_psa(frames_clean, STEPS)
r_3 = three_step(frames_clean[:3])
r_4 = four_step(frames_clean[:4])
r_5 = hariharan_five_step(frames_clean)
for name, r in [("3 pasos", r_3), ("4 pasos", r_4), ("Hariharan-5", r_5)]:
    d = demeaned_rms_error(r.phase, r_ls.phase)
    print(f"{name:14s} vs minimos cuadrados:  RMS = {d:.2e} rad")

In [ ]:
# Referencia limpia (campo sin bloque) y barrido de desintonia epsilon.
_, ref_plain = make_stack([0.0], geometry=geom_plain, illumination=illumination,
                          sensor=clean_sensor, use_irradiance=True, fringes=(1.0, 8.0),
                          seed0=99)
phi_true_plain = ref_plain.phase

def detuned(eps):
    applied = STEPS * (1.0 + eps)
    fr, _ = make_stack(applied, geometry=geom_plain, illumination=illumination,
                       sensor=clean_sensor, use_irradiance=True, fringes=(1.0, 8.0),
                       seed0=1000)
    return fr

eps_grid = np.linspace(0.0, 0.15, 11)
err_4 = [demeaned_rms_error(four_step(detuned(e)[:4]).phase, phi_true_plain) for e in eps_grid]
err_5 = [demeaned_rms_error(hariharan_five_step(detuned(e)).phase, phi_true_plain) for e in eps_grid]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(eps_grid * 100, np.array(err_4) * 1e3, "o-", label="4 pasos")
ax.plot(eps_grid * 100, np.array(err_5) * 1e3, "s-", label="Hariharan-5")
ax.set_xlabel(r"desintonia $\varepsilon$ del paso (%)")
ax.set_ylabel("error RMS de fase (mrad)")
ax.set_title("Robustez a la miscalibracion del paso")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("a epsilon = 10 %:  4 pasos =", f"{err_4[np.argmin(abs(eps_grid-0.10))]*1e3:.1f} mrad",
      " ; Hariharan-5 =", f"{err_5[np.argmin(abs(eps_grid-0.10))]*1e3:.1f} mrad")

**Lectura de la figura.** Sin desintonía ($\varepsilon=0$) los dos aciertan. Al crecer el error
del paso, el estimador de cuatro pasos se degrada **linealmente**, mientras que Hariharan-5 lo
hace de forma **cuadrática** (mucho más plano cerca del origen): su cero doble en la portadora
cancela el término lineal. Esta es la razón práctica por la que un interferómetro de bloques usa
cinco (o más) posiciones aunque el PZT no esté perfectamente calibrado.

## 6. El problema del desenrollado

La fase demodulada está **envuelta** en $(-\pi,\pi]$: en cada franja salta $2\pi$. Bajo muestreo
suficiente, la diferencia real entre píxeles vecinos es menor que $\pi$, así que el gradiente
verdadero se recupera **envolviendo** el gradiente observado, $\Delta\Phi=\operatorname{wrap}
(\Delta\varphi)$, e integrándolo se reconstruye $\Phi$ salvo una constante (pistón) [Servín *et
al.* (2014), cap. 6].

El problema solo es difícil donde el gradiente envuelto **no** es fiable —ruido, submuestreo,
píxeles inválidos—: ahí aparecen **residuos**, la carga $\pm 1$ de cada lazo $2\times 2$ cuya
integral $\oint\operatorname{wrap}(\nabla\varphi)$ no es cero. Alrededor de un residuo el
resultado **depende del camino**, y un barrido ingenuo propaga ese error local por todo el campo.
Los tres desenrolladores de la librería se distinguen por cómo tratan los residuos.

## 7. Residuos: dónde nace la dificultad

Se genera un *stack* **ruidoso** con la cámara real (`SensorModel.cs505mu()`) sobre un campo con
poca luz y fuerte caída de contraste en los bordes, se demodula, y se cuentan los residuos de la
fase limpia frente a la ruidosa.

In [ ]:
frames_noisy, _ = make_stack(STEPS, geometry=geom_plain, illumination=low_light,
                             sensor=camera, use_irradiance=False, fringes=(1.0, 8.0), seed0=7)
psi_noisy = hariharan_five_step(frames_noisy)

res_clean = find_residues(ref_plain.phase)
res_noisy = find_residues(psi_noisy.phase)
print("residuos en el campo limpio:", int(np.count_nonzero(res_clean)))
print("residuos en el campo ruidoso:", int(np.count_nonzero(res_noisy)))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
show(psi_noisy.phase, title=r"fase envuelta (ruidosa)", cmap="twilight", ax=axes[0])
show(psi_noisy.modulation, title="modulacion (calidad)", cmap="viridis", ax=axes[1], colorbar=True)
show(res_noisy, title="residuos (+1 / -1)", cmap="bwr", ax=axes[2], colorbar=True)
plt.tight_layout(); plt.show()

**Lectura de la figura.** El campo limpio no tiene ningún residuo; el ruidoso sí, y aparecen
justo donde el **mapa de modulación** cae (bordes con poca luz): baja confianza y gradiente poco
fiable van de la mano. Los residuos vienen en pares de signo opuesto (dipolos). Son los puntos
que un desenrollador debe rodear o aislar; su distribución explica por qué guiar la integración
por la calidad es tan efectivo.

## 8. Los tres desenrolladores

Sobre la **misma** fase ruidosa (campo continuo, sin bloque) se aplican los tres métodos:

- **Itoh** — integración por filas y columnas; baseline exacto en datos limpios, pero sigue un
  barrido fijo y no puede evitar los residuos.
- **Guiado por calidad** — *flood fill* que visita los píxeles en orden de modulación decreciente:
  integra primero por lo fiable.
- **Goldstein** — detecta residuos, los une con **cortes de rama** (*branch cuts*) y desenrolla
  sin cruzarlos, confinando la inconsistencia [Goldstein, Zebker & Werner, *Radio Sci.* **23**,
  713 (1988)].

In [ ]:
u_itoh = unwrap_itoh(psi_noisy.phase)
u_qual = unwrap_quality_guided(psi_noisy.phase, quality=psi_noisy.modulation)
u_gold, cuts = unwrap_goldstein(psi_noisy.phase, return_cuts=True)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
show(u_itoh, title="Itoh (barrido fijo)", cmap="viridis", ax=axes[0], colorbar=True)
show(u_qual, title="guiado por calidad", cmap="viridis", ax=axes[1], colorbar=True)
show(u_gold, title="Goldstein", cmap="viridis", ax=axes[2], colorbar=True)
axes[2].imshow(np.ma.masked_where(~cuts, cuts), cmap="autumn", interpolation="none")
plt.tight_layout(); plt.show()

print("cortes de rama de Goldstein:", int(cuts.sum()), "pixeles")

**Lectura de la figura.** Los tres reconstruyen la rampa global, pero difieren en los bordes
ruidosos: **Itoh** arrastra las inconsistencias a lo largo de filas y columnas (líneas espurias
que cruzan el campo); el **guiado por calidad** las contiene integrando por el interior fiable
primero; **Goldstein** dibuja los **cortes de rama** (en amarillo/rojo) que conectan los residuos
y su desenrollado no los cruza, dejando el resto del campo consistente. En una superficie con
pocos residuos aislados, guiado por calidad y Goldstein dan prácticamente el mismo resultado.

## 9. La segmentación es física: bloque y platina por separado

El borde bloque/platina **no** es un salto de envoltura: es una discontinuidad **real**, y el
**orden entero** de interferencia entre las dos superficies es genuinamente desconocido (lo
resuelve la determinación dimensional de L1 por fracciones exactas, no el desenrollador). Un
desenrollado de campo completo, obligado a ser continuo, **elige** un orden entero al cruzar el
borde —el más cercano— que no tiene por qué ser el físico. Segmentar con la máscara desenrolla
cada región por su cuenta y deja esa elección a L1.

In [ ]:
frames_step, truth_step = make_stack(STEPS, geometry=geometry, illumination=illumination,
                                     sensor=clean_sensor, use_irradiance=True, fringes=(0.0, 6.0),
                                     seed0=3)
psi_step = hariharan_five_step(frames_step)
block = truth_step.block_mask
platen = ~block

# Sin segmentar: un unico desenrollado sobre todo el campo.
naive = unwrap_quality_guided(psi_step.phase, quality=psi_step.modulation)

# Segmentado: cada region con su mascara (independientes, cada una su piston).
seg = np.full_like(psi_step.phase, np.nan)
seg[block] = unwrap_quality_guided(psi_step.phase, quality=psi_step.modulation, mask=block)[block]
seg[platen] = unwrap_quality_guided(psi_step.phase, quality=psi_step.modulation, mask=platen)[platen]

# Fase continua verdadera del simulador y alineacion de piston sobre la platina izquierda.
phi_true = (2.0 * np.pi / WAVELENGTH_NM) * truth_step.opd_nm
cols_block = np.where(block.any(axis=0))[0]
cb1, cb2 = cols_block[0], cols_block[-1]
left_platen = np.zeros_like(block); left_platen[:, :cb1] = True
naive_al = naive - np.nanmean(naive[left_platen]) + np.nanmean(phi_true[left_platen])
row = psi_step.phase.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
show(seg, title="segmentado (bloque | platina, cada uno su piston)",
     cmap="viridis", ax=axes[0], colorbar=True)
axes[0].contour(block, levels=[0.5], colors="r", linewidths=0.8)
axes[0].axhline(row, color="w", ls="--", lw=0.8)

cols = np.arange(psi_step.phase.shape[1])
ax = axes[1]
ax.plot(cols, phi_true[row], lw=2, label="fase continua verdadera")
ax.plot(cols, naive_al[row], lw=2, label="sin segmentar (un solo campo)")
ax.axvspan(cb1, cb2, color="gray", alpha=0.15, label="bloque")
ax.set_xlabel("columna"); ax.set_ylabel("fase (rad)")
ax.set_title(f"perfil en la fila {row}"); ax.legend(loc="best", fontsize=8)
plt.tight_layout(); plt.show()

step_true = 2.0 * np.pi * truth_step.fraction_truth
gap = float(np.nanmean(naive_al[block]) - np.nanmean(phi_true[block]))
print(f"escalon verdadero bloque-platina = {step_true:.3f} rad  (= 2*pi*F)")
print(f"desfase del campo sin segmentar en el bloque = {gap:.3f} rad  (~ -2*pi: orden entero inventado)")

In [ ]:
psi_step

In [ ]:
psi_step.phase

**Lectura de la figura.** A la izquierda, el desenrollado **segmentado**: bloque y platina, cada
uno con su propio pistón, quedan como superficies limpias e independientes (contorno rojo del
bloque). A la derecha, un perfil sobre una fila (la línea blanca): la **fase continua verdadera**
sube $2\pi F\approx 3{,}5$ rad al entrar al bloque, pero el desenrollado **sin segmentar**, al
imponer continuidad, envuelve ese escalón y coloca el bloque $2\pi$ por debajo: ha **inventado**
un orden entero. Ese entero es justamente lo que el método de fracciones exactas (L1) determina;
por eso el desenrollado no debe cruzar el borde bloque/platina.

## 10. Puente con L1: la fracción de franja

El paso métrico final no necesita el campo completo desenrollado: la **fracción de franja**
$F=\operatorname{frac}(2L/\lambda)$ sale de la diferencia de fase bloque$-$platina, tomada fila a
fila (el *tilt* común se cancela). `fraction_from_phase` lo hace sobre el mapa envuelto directo,
con dos regiones de interés que comparten filas. Se usa el campo limpio del bloque plano (§3).

In [ ]:
mask = truth.block_mask
cols_block = np.where(mask.any(axis=0))[0]
cb1, cb2 = cols_block[0], cols_block[-1]
rows = np.where(mask.any(axis=1))[0]
r1, r2 = rows[0] + 10, rows[-1] - 10

roi_block = (r1, r2, cb1 + 8, cb2 - 8)      # interior del bloque
roi_platen = (r1, r2, 8, cb1 - 8)           # banda de platina a la izquierda, mismas filas

F_hat = fraction_from_phase(psi.phase, roi_platen, roi_block)
F_true = truth.fraction_truth
gap = min(abs(F_hat - F_true) % 1.0, 1.0 - (abs(F_hat - F_true) % 1.0))

print(f"F recuperada = {F_hat:.6f}")
print(f"F verdadera  = {F_true:.6f}")
print(f"brecha       = {gap:.2e}  ({gap*WAVELENGTH_NM/2:.4f} nm en longitud, via dL = F*lambda/2)")

fig, ax = plt.subplots(figsize=(7, 3))
ax.imshow(psi.phase, cmap="twilight")
for (yy1, yy2, xx1, xx2), c, lab in [(roi_platen, "cyan", "platina"), (roi_block, "red", "bloque")]:
    ax.add_patch(plt.Rectangle((xx1, yy1), xx2 - xx1, yy2 - yy1, fill=False, ec=c, lw=1.5, label=lab))
ax.set_title(f"ROIs para la fraccion (F = {F_hat:.4f})"); ax.legend(loc="upper right")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

**Lectura de la figura.** Las dos regiones (platina en cian, bloque en rojo) comparten filas: al
restar sus fases fila a fila, la rampa de *tilt* se cancela y queda el corrimiento del bloque,
$2\pi F$. La fracción recuperada coincide con la verdad de terreno del simulador a nivel de ruido
numérico. Esta $F$ es exactamente la entrada del método de **fracciones exactas** (L1), que le
suma el orden entero $\kappa$ para obtener la longitud; ese paso queda fuera de este cuaderno.

## 11. Cierre

La cadena recorrida, de principio a fin:

$$\text{stack de }N\text{ interferogramas}\;\xrightarrow{\;\text{phase stepping}\;}\;
\varphi\ (\text{envuelta})\;\xrightarrow{\;\text{desenrollado}\;}\;\Phi\ (\text{continua})
\;\longrightarrow\; F=\operatorname{frac}(2L/\lambda).$$

- **Phase stepping** resuelve el problema inverso con una portadora temporal; el estimador de
  cinco pasos añade robustez a la desintonía del PZT.
- **Desenrollado** reconstruye la fase continua; guiado por calidad y Goldstein manejan los
  residuos que el ruido introduce, y la **máscara** respeta la segmentación física bloque/platina.
- El **puente con L1** lee la fracción sin necesidad de desenrollar el campo completo.

**Límites.** Todo con datos simulados; falta la validación con interferogramas reales del INM.
El hueco restante de la etapa franjas $\to$ fase es el **método de Fourier** de una sola toma.